# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiectul vostru.

---

## 1. Configurare — mai multe modele

In [ ]:
from openai import OpenAI

# ── Gemini ──────────────────────────────────────
GEMINI_KEY = "pune-cheia-ta"
gemini = OpenAI(
    api_key  = GEMINI_KEY,
    base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
)

# ── DeepSeek ─────────────────────────────────────
DEEPSEEK_KEY = "pune-cheia-ta"
deepseek = OpenAI(
    api_key  = DEEPSEEK_KEY,
    base_url = "https://api.deepseek.com"
)

# ── OpenRouter (acces la sute de modele cu un singur API key) ──
OPENROUTER_KEY = "pune-cheia-ta"
openrouter = OpenAI(
    api_key  = OPENROUTER_KEY,
    base_url = "https://openrouter.ai/api/v1"
)

print("Clienți configurați.")

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [ ]:
def intreaba(client, model, prompt, system=None, temperature=0.7):
    """Trimite un prompt la orice model și returnează răspunsul."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    raspuns = client.chat.completions.create(
        model       = model,
        messages    = messages,
        temperature = temperature,
        max_tokens  = 300
    )
    return raspuns.choices[0].message.content.strip()


# Modelele pe care le testăm
MODELE = [
    (gemini,    "gemini-2.5-flash",          "Gemini 2.5 Flash"),
    (deepseek,  "deepseek-chat",             "DeepSeek Chat"),
    (openrouter,"meta-llama/llama-3.1-8b-instruct:free", "Llama 3.1 8B (gratuit)"),
]

print("Modele pregătite:", [m[2] for m in MODELE])

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [ ]:
PROMPT_RO = """
Rezumă în 2 propoziții, în română, ce s-a întâmplat în politica românească în ultimii ani.
Fii factual și neutru.
"""

print("=" * 60)
for client, model_id, nume in MODELE:
    print(f"\n[ {nume} ]")
    print(intreaba(client, model_id, PROMPT_RO))
    print()

## 4. Test 2 — Urmează instrucțiunile din system prompt?

Vedem dacă modelele respectă rolul dat prin `system`.

In [ ]:
SYSTEM = "Ești un comentator politic populist. Răspunzi emoțional, folosești cuvinte simple, apelezi la popor."
PROMPT = "Ce crezi despre politicienii din România?"

print("=" * 60)
for client, model_id, nume in MODELE:
    print(f"\n[ {nume} ]")
    print(intreaba(client, model_id, PROMPT, system=SYSTEM, temperature=0.9))
    print()

## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [ ]:
import json

PROMPT_JSON = """
Analizează acest comentariu și returnează DOAR un JSON valid, fără explicații:
Comentariu: "Acești politicieni corupți distrug țara! Poporul suferă în timp ce ei fură!"

Format JSON dorit:
{
  "ton": "pozitiv/negativ/neutru",
  "emotie_dominanta": "string",
  "subiect": "string",
  "populism": true/false
}
"""

print("=" * 60)
for client, model_id, nume in MODELE:
    print(f"\n[ {nume} ]")
    raspuns = intreaba(client, model_id, PROMPT_JSON, temperature=0.1)
    print("Raw:", raspuns)
    # Incercam sa parsam JSON-ul
    try:
        # Curatam backticks daca exista
        curat = raspuns.strip().strip("```json").strip("```").strip()
        date  = json.loads(curat)
        print("Parsed OK:", date)
    except Exception as e:
        print("Eroare JSON:", e)
    print()

## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [ ]:
PROMPT_STAB = "Curtea Constituțională a anulat alegerile. Ce crezi?"

print("=" * 60)
print("[ Gemini — același prompt, temperature diferite ]\n")

for temp in [0.1, 0.7, 1.2]:
    r = intreaba(gemini, "gemini-2.5-flash", PROMPT_STAB, temperature=temp)
    print(f"temperature={temp}:")
    print(r)
    print()

## 7. Matricea de decizie — alegeți modelul pentru proiect

Completați tabelul de mai jos cu observațiile voastre din testele de mai sus.

| Criteriu | Gemini 2.5 Flash | DeepSeek Chat | Llama 3.1 8B |
|---|---|---|---|
| Calitate română | ✓ / ✗ | ✓ / ✗ | ✓ / ✗ |
| Urmează system prompt | ✓ / ✗ | ✓ / ✗ | ✓ / ✗ |
| Produce JSON valid | ✓ / ✗ | ✓ / ✗ | ✓ / ✗ |
| Stabilitate (temperature) | ✓ / ✗ | ✓ / ✗ | ✓ / ✗ |
| Cost / free tier | gratuit | gratuit | gratuit |
| Viteză | rapidă | rapidă | rapidă |

**Model ales pentru proiect:** _______________  
**Fallback (dacă primul nu merge):** _______________  
**Temperature setată:** _______________  
**Motivare (1-2 propoziții):** _______________

## 8. Configurația finală a proiectului

In [ ]:
# Copiați această celulă în .env sau în config.py al proiectului

MODEL_PRINCIPAL = "gemini-2.5-flash"  # TODO: schimbați cu modelul ales
MODEL_FALLBACK  = "deepseek-chat"     # TODO: schimbați cu fallback-ul ales
TEMPERATURE     = 0.7                 # TODO: ajustați după testele voastre

print("Configurație proiect:")
print(f"  Model principal : {MODEL_PRINCIPAL}")
print(f"  Fallback        : {MODEL_FALLBACK}")
print(f"  Temperature     : {TEMPERATURE}")

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales